In [233]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.head()

,id,name,make,model,year,price,engine,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain
0,1,2024 INFINITI QX50 SPORT,INFINITI,QX50,2024,49404.0,ER,4.0,Gasoline,7.0,(CVT) CONT VAR.,SPORT,NaN,4.0,Radiant White,Monaco Red,Front-wheel Drive
1,2,2024 Mazda CX-90 PHEV Base,Mazda,CX-90 PHEV,2024,55399.0,"TIV-G 2.5L I-4 gasoline direct injection, DOHC...",4.0,Gasoline,2.0,8-Speed Automatic,Base,SUV,4.0,Soul Red Crystal Metallic,Griege,All-wheel Drive
2,3,2024 Honda CR-V Hybrid Sport Touring,Honda,CR-V Hybrid,2024,42005.0,16V GDI DOHC Hybrid,4.0,Hybrid,1.0,Automatic CVT,Sport Touring,SUV,4.0,Radiant Red Metallic,Black,All-wheel Drive
3,4,2024 Jeep Grand Cherokee L Limited,Jeep,Grand Cherokee L,2024,52567.0,OHV,6.0,Gasoline,246.0,8-Speed Automatic,Limited,SUV,4.0,Velvet Red Pearlcoat,Global Black,Four-wheel Drive
4,5,2024 Acura MDX Technology,Acura,MDX,2024,56800.0,24V GDI SOHC,6.0,Gasoline,5.0,10-Speed Automatic,Technology,SUV,4.0,Gray,NaN,Front-wheel Drive


In [234]:
mean_1 = df_train[df_train.cylinders == 0.0].price.mean()
mean_2 = df_train[df_train.cylinders == 3.0].price.mean()
mean_3 = df_train[df_train.cylinders == 4.0].price.mean()
mean_4 = df_train[df_train.cylinders == 6.0].price.mean()
mean_5 = df_train[df_train.cylinders == 8.0].price.mean()

means = [mean_1, mean_2, mean_3, mean_4, mean_5]

subtask_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": [1, 1],
    "answer": max(means)
})

subtask_1.iloc[0]

subtaskID          1.0
datapointID        1.0
answer         66045.0
Name: 0, dtype: float64

In [235]:
#preprocessinguuhhhh
#Label encooding and then imputing missing values and then scaling everything
#make, model, engine, fuel, transmission, trim, body, ext color, int color, drivetrain need encoding

from sklearn.preprocessing import LabelEncoder

encoding_cols = ['make', 'model', 'engine', 'fuel', 'transmission', 'trim', 'body', 'exterior_color', 'interior_color', 'drivetrain']

for column in encoding_cols:
    enc = LabelEncoder()
    enc.fit(pd.concat([df_train[column], df_test[column]]))
    df_train[column] = enc.transform(df_train[column])
    df_test[column] = enc.transform(df_test[column])

imputing_cols = ['price', 'cylinders', 'mileage', 'doors']

for column in imputing_cols:
    meanie = df_train[column].mean()
    df_train[column] = df_train[column].fillna(meanie)

features = ['make', 'model', 'year', 'engine', 'cylinders', 'fuel', 'mileage', 'transmission', 'trim', 'body', 'doors', 'exterior_color', 'interior_color', 'drivetrain']
target = 'price'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_final = pd.DataFrame(scaler.transform(X_final), columns=features)

X.head()

,make,model,year,engine,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain
0,-0.018437,0.524410,0.287553,0.865798,-0.767749,0.197665,-0.110592,-1.873282,0.702611,1.748890,0.215746,0.754209,1.474169,1.243975
1,1.048601,-1.144821,0.287553,1.073382,-0.767749,0.197665,-0.121667,-0.085080,-1.171356,0.277878,0.215746,1.221967,0.683000,-0.883249
2,-0.323304,-1.214373,0.287553,-1.002462,-0.767749,1.014817,-0.123883,1.232543,1.109995,0.277878,0.215746,0.728924,-0.741104,-0.883249
3,0.286431,-0.287022,0.287553,1.038785,0.790099,0.197665,0.418823,-0.085080,-0.193634,0.277878,0.215746,1.563305,0.445649,0.180363
4,-2.000077,0.060735,0.287553,-0.414306,0.790099,0.197665,-0.115022,-1.590935,1.313687,0.277878,0.215746,-0.156025,2.660922,1.243975


In [236]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [237]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

model = RandomForestRegressor(random_state = 42)
model.fit(X, y)
y_preds = model.predict(X_test)
r2_score(y_test, y_preds)

0.9782795785961201

In [238]:
y_final = model.predict(X_final)

In [239]:
subtask_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.id,
    "answer": y_final
})

submission = pd.concat([subtask_1, subtask_2])

submission = submission.iloc[1:]
submission

,subtaskID,datapointID,answer
1,1,1,66045.000000
0,2,803,47420.019524
1,2,804,41025.004442
2,2,805,29844.340000
3,2,806,36806.332221
...,...,...,...
194,2,997,38224.890000
195,2,998,48934.510714
196,2,999,34128.324127
197,2,1000,63172.320000


In [240]:
submission.to_csv("submission.csv", index=False)